# ZAIC 2026 LifeMate — v5: train LLM chịu được lỗi ASR

v4 (public **44.46**): Whisper fine-tune đưa WER 34.6% → 11.5%. Nhưng vẫn còn **14 điểm** mất do ASR
(val_unseen: transcript gold 49.0 vs ASR 35.1). Vòng này dạy LLM chịu được lỗi nhận dạng.

**Cách làm:** không thể dùng transcript của `asr_ft_v4` để train LLM, vì ASR đó đã học thuộc chính các clip train
(transcript sẽ gần như hoàn hảo, không có nhiễu để học). Nên notebook train **2 Whisper fold**, mỗi fold nhận dạng
nửa dữ liệu mà nó chưa thấy — đó mới là nhiễu thật.

**GPU: A100** (L4 mất khoảng 4–5 giờ, A100 khoảng 2 giờ). Upload `lifemate_baseline.zip` mới → Run all →
nộp `submission_public_v5.jsonl` (**small models (<6B)**) → gửi `results_v5.zip` + điểm.

Nếu đã có dữ liệu synthetic của teammate: đặt các file `*.jsonl` vào `MyDrive/ZAIC/synth_specs/`, notebook tự gộp.

In [ ]:
# ==== CẤU HÌNH ====
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ASR_BASE   = "openai/whisper-large-v3-turbo"
ASR_EPOCHS = 3
LLM_EPOCHS = 1
BATCH      = 32
RUN_FOLDS  = True     # train 2 Whisper fold để lấy transcript nhiễu (bỏ qua nếu đã có asr_train_oof.json trong Drive)
DATA_URL   = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR  = "/content/drive/MyDrive/ZAIC"
ASR_PROD   = DRIVE_DIR + "/asr_ft_v4"          # ASR của v4, dùng khi chạy thật
WORK, DATA_RAW, RES = "/content/work", "/content/data_raw", "/content/results_v5"
LLM_OUT    = "/content/ft_v5"

In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU"
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0); print(GPU_NAME, DTYPE)
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob, subprocess, json, time
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(os.path.join(DRIVE_DIR, "lifemate_baseline.zip")).extractall(WORK)
assert os.path.exists(f"{WORK}/lifemate/synth_build.py"), "lifemate_baseline.zip trong Drive là bản cũ -> upload bản mới"
assert os.path.exists(ASR_PROD + "/config.json"), "Không thấy ASR v4 trong Drive: " + ASR_PROD
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        assert subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL]).returncode == 0, "Tải dữ liệu lỗi"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA:", DATA)

In [ ]:
%%capture pipout
!pip install -q vllm peft soundfile
!pip uninstall -y torchao

In [ ]:
print(pipout.stderr[-800:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers|peft) " | tee {RES}/versions.txt
%cd {WORK}
ids = [json.loads(l)["id"] for l in open(f"{DATA}/train/train.jsonl", encoding="utf-8")]
open("splits/foldA.txt", "w").write("\n".join(ids[0::2]))   # mẫu chẵn
open("splits/foldB.txt", "w").write("\n".join(ids[1::2]))   # mẫu lẻ
!cat splits/val_unseen.txt splits/val_seen.txt > splits/val_all.txt
print(len(ids), "mẫu train")

## 1. Hai Whisper fold → transcript nhiễu thật cho toàn bộ train (~70 phút A100)

In [ ]:
%cd {WORK}
OOF = os.path.join(DRIVE_DIR, "asr_train_oof.json")
if RUN_FOLDS and not os.path.exists(OOF):
    part = {}
    for f, other in [("A", "B"), ("B", "A")]:
        # train trên fold f (loại fold kia), rồi nhận dạng fold kia
        !python -m lifemate.asr_finetune --root {DATA} --exclude splits/fold{other}.txt --model {ASR_BASE} --out /content/asr_fold{f} --epochs {ASR_EPOCHS} 2>&1 | grep -E "^train clips|^step (50|150|250|350)/|^saved"
        !python -m lifemate.asr --root {DATA} --split train --ids splits/fold{other}.txt --model /content/asr_fold{f} --dtype {DTYPE} --out oof_{other}.json 2>&1 | grep -E "^model=|^WER"
        part.update(json.load(open(f"oof_{other}.json")))
        shutil.copy(f"oof_{other}.json.wer.txt", f"{RES}/wer_oof_{other}.txt")
        shutil.rmtree(f"/content/asr_fold{f}", ignore_errors=True)   # giải phóng đĩa
    json.dump(part, open(OOF, "w"), ensure_ascii=False)
    print("OOF transcripts:", len(part))
shutil.copy(OOF, "asr_train_oof.json"); print("OOF clips:", len(json.load(open(OOF))))

## 2. Dữ liệu SFT: mỗi mẫu 3 bản (gold / đổi tên app / transcript nhiễu) + synthetic nếu có

In [ ]:
%cd {WORK}
SPEC_DIR = os.path.join(DRIVE_DIR, "synth_specs")
specs = sorted(glob.glob(SPEC_DIR + "/*.jsonl"))
print("synth spec files:", len(specs))
parts = []
!python -m lifemate.sft_data --root {DATA} --ids splits/train_fit.txt --aug-copies 1 --asr asr_train_oof.json --out sft_fit.jsonl
parts.append("sft_fit.jsonl")
if specs:
    !python -m lifemate.synth_build --root {DATA} --tools-from public_test --specs {SPEC_DIR}/*.jsonl 2>&1 | tail -20
    !python -m lifemate.sft_data --root {DATA} --split synth --aug-copies 1 --out sft_synth.jsonl
    parts.append("sft_synth.jsonl")
with open("sft_all.jsonl", "w", encoding="utf-8") as out:
    for p in parts:
        out.writelines(open(p, encoding="utf-8"))
n = sum(1 for _ in open("sft_all.jsonl")); print("tổng ví dụ SFT:", n)

## 3. Train LLM (A100 khoảng 1–1.5 giờ). Theo dõi `loss` / `eta`

In [ ]:
%cd {WORK}
!python -m lifemate.train_lora --data sft_all.jsonl --model {BASE_MODEL} --out {LLM_OUT} --epochs {LLM_EPOCHS} --detach-prefix 2>&1 | grep -vi warn
assert os.path.exists(f"{LLM_OUT}/merged/config.json"), "train lỗi: xem log phía trên"
shutil.copy(f"{LLM_OUT}/train_log.txt", f"{RES}/train_log.txt")
shutil.copytree(f"{LLM_OUT}/adapter", os.path.join(DRIVE_DIR, "ft_v5_adapter"), dirs_exist_ok=True)

## 4. Đánh giá trên val, dùng ASR thật (v4: 35.06 / 54.11; trần transcript gold: 49.0 / 67.7)

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split train --ids splits/val_all.txt --model {ASR_PROD} --dtype {DTYPE} --out asr_val.json 2>&1 | grep -E "^model=|^WER"
sc = {}
for S in ["val_unseen", "val_seen"]:
    !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch {BATCH} --transcripts asr_val.json --out runs/v5_{S} 2>&1 | tail -1
    !python scripts/score.py --root {DATA} --split train --pred runs/v5_{S}/submission.jsonl --by-tag > {RES}/score_v5_{S}.txt
    !python -m lifemate.report --root {DATA} --split train --pred runs/v5_{S}/submission.jsonl --show 6 > {RES}/report_v5_{S}.txt
    sc[S] = float(open(f"{RES}/score_v5_{S}.txt").readline().split()[-1])
    !head -10 {RES}/report_v5_{S}.txt
print("v5:", sc, "| v4: {'val_unseen': 35.06, 'val_seen': 54.11}")
json.dump(sc, open(f"{RES}/e2e_v5.json", "w"))

## 5. Public → `submission_public_v5.jsonl`

In [ ]:
%cd {WORK}
!python -m lifemate.asr --root {DATA} --split public_test --model {ASR_PROD} --dtype {DTYPE} --out asr_public_v5.json 2>&1 | tail -1
!python -m lifemate.run --root {DATA} --split public_test --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch {BATCH} --transcripts asr_public_v5.json --out runs/public_v5 2>&1 | tail -1
shutil.copy("runs/public_v5/submission.jsonl", f"{RES}/submission_public_v5.jsonl")
shutil.copy("runs/public_v5/submission.jsonl", os.path.join(DRIVE_DIR, "submission_public_v5.jsonl"))
shutil.copy("runs/public_v5/traces.jsonl", f"{RES}/traces_public_v5.jsonl")
%cd {DATA}/public_test
!python -m zaic_tools --split public_test --root .. --validate {RES}/submission_public_v5.jsonl 2>&1 | tail -3 | tee {RES}/validate_public_v5.txt

## 6. Latency batch 1 (luật: ≤ 30 s/mẫu trên A30) và đóng gói

In [ ]:
%cd {WORK}
!head -30 splits/val_unseen.txt > splits/lat30.txt
!python -m lifemate.run --root {DATA} --split train --ids splits/lat30.txt --backend vllm --model {LLM_OUT}/merged --dtype {DTYPE} --batch 1 --transcripts asr_val.json --out runs/lat 2>&1 | tail -1
secs = sorted(json.loads(l)["seconds"] for l in open("runs/lat/traces.jsonl"))
s = f"GPU={GPU_NAME} batch=1 (chưa gồm ASR) n={len(secs)} mean={sum(secs)/len(secs):.2f}s max={secs[-1]:.2f}s"
print(s); open(f"{RES}/latency.txt", "w").write(s + "\n")
open(f"{RES}/config.txt", "w").write(f"BASE={BASE_MODEL}\nLLM_EPOCHS={LLM_EPOCHS}\nSFT={n}\nsynth_specs={len(specs)}\nGPU={GPU_NAME}\n")
z = shutil.make_archive("/content/results_v5", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results_v5.zip")); print("saved results_v5.zip")
from google.colab import files
files.download(z); files.download(f"{RES}/submission_public_v5.jsonl")